### 1.5 Data Preparation and Pre-processing

The Exploratory Data Analysis (EDA) revealed that the raw Sentiment140 dataset contains significant linguistic noise, including URLs, user mentions, and non-informative stop words. This phase constructs a comprehensive Natural Language Processing (NLP) pipeline to clean, normalize, and tokenize the text, transforming the unstructured tweets into a refined corpus suitable for training supervised machine learning models.

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

# Define column names and load the dataset
column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
file_path = 'training.1600000.processed.noemoticon.csv'

print("Loading dataset...")
df = pd.read_csv(file_path, names=column_names, encoding='latin-1')

# To optimize memory and processing time for this assignment, we will drop unnecessary columns
df = df[['target', 'text']]

# Remap target labels to binary (0 = Negative, 1 = Positive) for easier model training later
df['target'] = df['target'].map({0: 0, 4: 1})

print(f"Dataset loaded. Shape: {df.shape}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Damien\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Damien\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading dataset...
Dataset loaded. Shape: (1600000, 2)


#### 1.5.1 Text Normalization and Noise Removal
Social media text requires aggressive filtering. The following function utilizes Regular Expressions (`re`) to systematically eliminate platform-specific artifacts. Specifically, it strips out URLs, removes Twitter handles (words starting with '@'), eliminates all punctuation and special characters, and converts the entire string to lowercase to ensure uniform feature extraction.

In [2]:
def clean_tweet_text(text):
    # 1. Convert text to lowercase
    text = text.lower()
    
    # 2. Remove URLs (http:// or https://)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Remove user mentions (@username)
    text = re.sub(r'@\w+', '', text)
    
    # 4. Remove HTML entities (like &amp;)
    text = re.sub(r'&\w+;', '', text)
    
    # 5. Remove punctuation, numbers, and special characters (keep only alphabets)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 6. Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Applying regex cleaning pipeline. This may take a few minutes for 1.6 million rows...")
# Apply the cleaning function to the text column
df['cleaned_text'] = df['text'].apply(clean_tweet_text)

print("Cleaning complete. Previewing transformed text:")
display(df[['text', 'cleaned_text']].head())

Applying regex cleaning pipeline. This may take a few minutes for 1.6 million rows...
Cleaning complete. Previewing transformed text:


,text,cleaned_text
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww thats a bummer you shoulda got david carr...
1,is upset that he can't update his Facebook by ...,is upset that he cant update his facebook by t...
2,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,"@nationwideclass no, it's not behaving at all....",no its not behaving at all im mad why am i her...


#### 1.5.2 Tokenization and Stop Word Filtering
Once the text is normalized, it must be broken down into discrete tokens. During this phase, standard English stop words (e.g., "the", "is", "in") are removed using the NLTK corpus. These highly frequent words carry negligible emotional weight and their removal significantly reduces the dimensionality of the final feature space, improving computational efficiency during model training.

In [3]:
# Establish the set of English stop words
stop_words = set(stopwords.words('english'))

def tokenize_and_filter(text):
    # Tokenize the cleaned string
    tokens = word_tokenize(text)
    
    # Filter out stop words and single-character tokens (like stray 'u' or 'n')
    filtered_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    
    # Rejoin the tokens into a single clean string for the vectorizer
    return ' '.join(filtered_tokens)

print("Applying tokenization and stop word filtering...")
df['final_text'] = df['cleaned_text'].apply(tokenize_and_filter)

# Drop any rows that became entirely empty after cleaning (e.g., a tweet that was just a URL)
df = df.dropna(subset=['final_text'])
df = df[df['final_text'].str.strip() != '']

print("Filtering complete. Previewing final feature-ready text:")
display(df[['cleaned_text', 'final_text']].head())

Applying tokenization and stop word filtering...
Filtering complete. Previewing final feature-ready text:


,cleaned_text,final_text
0,awww thats a bummer you shoulda got david carr...,awww thats bummer shoulda got david carr third...
1,is upset that he cant update his facebook by t...,upset cant update facebook texting might cry r...
2,i dived many times for the ball managed to sav...,dived many times ball managed save rest go bounds
3,my whole body feels itchy and like its on fire,whole body feels itchy like fire
4,no its not behaving at all im mad why am i her...,behaving im mad cant see


#### 1.5.3 Exporting the Processed Corpus
To prevent having to rerun this computationally expensive NLP pipeline during the model training phase, the finalized, cleaned dataset is exported to a new CSV file. This file will serve as the direct input for feature extraction (TF-IDF vectorization) and model training in Section 2.

In [4]:
output_file = 'sentiment140_cleaned.csv'

print(f"Exporting processed dataset to {output_file}...")
# Save only the necessary columns to keep the file size manageable
df[['target', 'final_text']].to_csv(output_file, index=False)

print("Export successful. Data preparation phase complete.")

Exporting processed dataset to sentiment140_cleaned.csv...
Export successful. Data preparation phase complete.
